In [3]:
!source ../wcenv/bin/activate

In [17]:
!pip install -e ..

Obtaining file:///home/sagar/winogender_contextuality
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for winogender_contextuality (pyproject.toml) ... done
  Created wheel for winogender_contextuality: filename=winogender_contextuality-0.0.1-py3-none-any.whl size=3635 sha256=8f83f06750999f9d09cc47523a41c6a1871607f6546726d10fea1df95c59f256
  Stored in directory: /tmp/pip-ephem-wheel-cache-qr822z1f/wheels/f6/b9/38/03ac5a5ccd63b90faa34c1614fd3e708a9c34ab8edb44270e9
Successfully built winogender_contextuality
  Attempting uninstall: winogender_contextuality
    Found existing installation: winogender_contextuality 0.0.1
    Uninstalling winogender_contextuality-0.0.1:
      Successfully uninstalled winogender_contextuality-0.0.1


In [4]:
!pip install -r /home/sagar/winogender_contextuality/requirements.txt

Obtaining file:///home/sagar/winogender_contextuality/notebooks (from -r /home/sagar/winogender_contextuality/requirements.txt (line 16))
ERROR: file:///home/sagar/winogender_contextuality/notebooks (from -r /home/sagar/winogender_contextuality/requirements.txt (line 16)) does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.


In [2]:
torch.version.cuda

'12.6'

In [1]:
pip show winogender_contextuality

Name: winogender_contextuality
Version: 0.0.1
Summary: Assessing contextuality in generative language models' resolution of gendered pronouns.
Home-page: 
Author: Sagar Kumar
Author-email: 
License: 
Location: /home/sagar/miniconda3/lib/python3.13/site-packages
Editable project location: /home/sagar/winogender_contextuality
Requires: 
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [1]:
import winogender_contextuality as wc
from winogender_contextuality.modeling.ModelProbs import *
from winogender_contextuality.modeling.contextuality import *

2025-07-24 21:10:11.717 | INFO     | winogender_contextuality.config:<module>:13 - PROJ_ROOT path is: /home/sagar/winogender_contextuality
2025-07-24 21:10:11.718 | INFO     | winogender_contextuality.config:<module>:17 - DATA_ROOT path is: /data_users1/sagar/winogender_contextuality


2025-07-24 21:10:17.125 | INFO     | winogender_contextuality.modeling.run_local:<module>:9 - torch available: True
2025-07-24 21:10:17.151 | INFO     | winogender_contextuality.modeling.run_local:<module>:10 - 12.6
2025-07-24 21:10:17.190 | INFO     | winogender_contextuality.modeling.run_local:<module>:12 - _CudaDeviceProperties(name='NVIDIA TITAN Xp', major=6, minor=1, total_memory=12188MB, multi_processor_count=30, uuid=69e3ceae-9a8c-607c-5cd7-cde99d6659aa, L2_cache_size=3MB)


In [62]:
from pathlib import Path
import os
from matplotlib import pyplot as plt
import pandas as pd
import ast
import importlib
#import torch

## Loading Model

In [3]:
HF_KEY = ""

In [4]:
wc.config.MODELS_DIR

PosixPath('/data_users1/sagar/winogender_contextuality/models')

In [5]:
mp = wc.modeling.ModelProbs.ModelProbs(mode='gpu', 
                                       model_name="meta-llama/Llama-3.2-1B-Instruct", 
                                       key=HF_KEY, 
                                       model_path=wc.config.MODELS_DIR)

In [6]:
mp.load_model()

2025-07-24 21:10:23.184 | INFO     | winogender_contextuality.modeling.run_local:load_model:33 - Connected with Huggingface
2025-07-24 21:10:23.184 | INFO     | winogender_contextuality.modeling.run_local:load_model:43 - Loading quantized model


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

2025-07-24 21:10:33.764 | INFO     | winogender_contextuality.modeling.run_local:load_model:50 - Model cached in /data_users1/sagar/winogender_contextuality/models/cache
2025-07-24 21:10:33.765 | INFO     | winogender_contextuality.modeling.ModelProbs:load_model:55 - Loaded model meta-llama/Llama-3.2-1B-Instruct to /data_users1/sagar/winogender_contextuality/models


## Loading Data

In [57]:
from winogender_contextuality.modeling.prompting import *
from winogender_contextuality.utils import *
from itertools import chain

In [15]:
pairs = pd.read_csv(wc.config.PROCESSED_DATA_DIR / "wino_pairs.tsv", sep="\t")

In [38]:
pairs.head()

,template_1,differences_1,referent_1,template_2,differences_2,referent_2
0,The technician told the customer that BLANK co...,"['he', 'she']",customer,The technician told the customer that BLANK ha...,"['he', 'she']",technician
1,The technician told the customer that BLANK co...,"['he', 'she']",customer,The technician told the customer that BLANK ha...,"['he', 'she']",technician
2,The technician told the customer that BLANK co...,"['he', 'she']",customer,The technician told the customer that BLANK ha...,"['he', 'she']",technician
3,The technician told the customer that BLANK co...,"['he', 'she']",customer,The technician told the customer that BLANK ha...,"['he', 'she']",technician
4,The technician told the customer that BLANK co...,"['he', 'she']",customer,The technician told the customer that BLANK ha...,"['he', 'she']",technician


# Test Prompts

In [20]:
test_prompt = wc.modeling.prompting.get_role_content_prompt(game=False, options=pairs.differences_1[0], 
                                      sentence=pairs.template_1[0])

In [24]:
test_logits = mp.get_raw_logits(prompt=test_prompt).to('cpu')

In [35]:
mp.get_token_ids(options=[" "+s for s in ast.literal_eval(pairs.differences_1[0])])

[[568], [1364]]

In [36]:
[" "+s for s in ast.literal_eval(pairs.differences_1[0])]

[' he', ' she']

In [37]:
test_logits[568], test_logits[1364]

(tensor(20.7344, dtype=torch.float16), tensor(21.2500, dtype=torch.float16))

In [39]:
test_prompt_2 = wc.modeling.prompting.get_role_content_prompt(game=False, options=pairs.differences_2[0], 
                                      sentence=pairs.template_2[0])
test_logits_2 = mp.get_raw_logits(prompt=test_prompt_2).to('cpu')
test_logits_2[568], test_logits_2[1364]

(tensor(19.7500, dtype=torch.float16), tensor(20.1250, dtype=torch.float16))

In [46]:
probs1 = masked_softmax([568, 1364], test_logits)
probs1 = probs1/torch.sum(probs1) # Normalized

In [73]:
probs1.detach().numpy()

array([0.3738, 0.626 ], dtype=float16)

In [42]:
0.3738 + 0.6260

0.9998

## Contextuality

In [ ]:
# In the CSV, "he first" is always the original direction and then it is reversed in the feminine case

In [52]:
# Probably need to output the tokenized prompt from the function, not the list OR input the tokenized prompt here
ms = MeasurementScenario(observations=['template_1', 'template_2'], 
                         measurements=['he_first', 'she_first'],
                         outcomes=[0,1]) # 1 corresponds to selecting the feminine pronoun

In [53]:
ms.scenario

<xarray.DataArray (context_pair: 4, outcome_pair: 4)> Size: 128B
array([[0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.]])
Coordinates:
  * context_pair  (context_pair) int64 32B 0 1 2 3
  * outcome_pair  (outcome_pair) int64 32B 0 1 2 3

In [82]:
ms.outcome_pairs

[(0, 0), (0, 1), (1, 0), (1, 1)]

In [97]:
def reverse_pronouns(options,
                     measurement, 
                     prime='she_first'):
    
    pronouns = ast.literal_eval(options)
    if measurement==prime:
        return pronouns[::-1]
    else:
        return pronouns

In [101]:
# This is gonna be the function 

for arr_idx, pair in enumerate(ms.context_pairs):
    arr = np.zeros((len(ms.observations), len(ms.outcomes)))
    for pair_idx, oc_idx in enumerate(pair):
        oc_pair = ms.reverse_context_idx.get(oc_idx)
        obs, ctx = oc_pair
        obs_index = obs[-1]
        sent = pairs[obs][0] # the 0 is iterated index
        pnouns = reverse_pronouns(pairs[f"differences_{obs_index}"][0], ctx) 
        prompt = wc.modeling.prompting.get_role_content_prompt(game=False, options=pnouns, sentence=sent)
        logits = mp.get_raw_logits(prompt=prompt).to('cpu')
        # For now, we just use the two tokens 
        tokens = mp.get_token_ids(options=[" "+s for s in ast.literal_eval(pairs[f"differences_{obs_index}"][0])]) # in order [m, f]
        softmax = masked_softmax(list(chain.from_iterable(tokens)), logits)
        probs = softmax/torch.sum(softmax)
        arr[pair_idx] = probs.detach().numpy()
    ms.scenario[arr_idx] = arr.reshape(-1) # does this work


In [102]:
ms.scenario

<xarray.DataArray (context_pair: 4, outcome_pair: 4)> Size: 128B
array([[0.39233398, 0.60742188, 0.41113281, 0.58886719],
       [0.39233398, 0.60742188, 0.15405273, 0.84570312],
       [0.07922363, 0.92089844, 0.41113281, 0.58886719],
       [0.07922363, 0.92089844, 0.15405273, 0.84570312]])
Coordinates:
  * context_pair  (context_pair) int64 32B 0 1 2 3
  * outcome_pair  (outcome_pair) int64 32B 0 1 2 3

In [103]:
ms.incidence_matrix()

<xarray.DataArray (s: 16, t: 16)> Size: 2kB
array([[1., 1., 0., 0., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 1., 1., 0., 0., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 0., 0., 1., 1., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 1., 0., 0., 1., 1.],
       [1., 0., 1., 0., 1., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 1., 0., 1., 0., 1., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 1., 0., 1., 0., 1., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 1., 0., 1., 0., 1.],
       [1., 1., 0., 0., 0., 0., 0., 0., 1., 1., 0., 0., 0., 0., 0., 0.],
       [0., 0., 1., 1., 0., 0., 0., 0., 0., 0., 1., 1., 0., 0., 0., 0.],
       [0., 0., 0., 0., 1., 1., 0., 0., 0., 0., 0., 0., 1., 1., 0., 0.],
       [0., 0., 0., 0., 0., 0., 1., 1., 0., 0., 0., 0., 0., 0., 1., 1.],
       [1., 0., 1., 0., 0., 0., 0., 0., 1., 0., 1., 0., 0., 0., 0., 0.],
       [0., 1., 0., 1., 0., 0., 0., 0., 0., 1., 0., 1., 0., 0., 0., 0.],
       [0., 0., 0., 0., 1., 0., 1., 0., 0., 0., 0., 0., 1., 0., 1., 0.],
       [0., 0., 0., 0., 0., 1., 0., 1., 0., 0., 0., 0., 0., 1., 0., 1.]])
Coordinates:
  * s        (s) int64 128B 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15
  * t        (t) int64 128B 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15

In [104]:
check_feasibility(ms)

2025-07-25 14:20:36.756 | INFO     | winogender_contextuality.modeling.contextuality:check_feasibility:106 - Measurement status: 2


(False,
        message: The problem is infeasible. (HiGHS Status 8: model_status is Infeasible; primal_status is None)
        success: False
         status: 2
            fun: None
              x: None
            nit: 9
          lower:  residual: None
                 marginals: None
          upper:  residual: None
                 marginals: None
          eqlin:  residual: None
                 marginals: None
        ineqlin:  residual: None
                 marginals: None)